# Mizan · Fine-tuning a specialist Arabic model

Fine-tune **AraBERT** (`aubmindlab/bert-base-arabertv02`, an open-source Arabic BERT) on 1,078 Arabic banking queries, then:

1. score it on the same 370 test queries as the baselines (overall, MSA and dialect),
2. measure **single-query latency on CPU**, the realistic case for a model served inside a bank's own infrastructure,
3. choose a **confidence threshold** below which a request is routed to a human agent,
4. optionally publish the model to the Hugging Face Hub so the API and demo can load it.

**Runtime:** Runtime → Change runtime type → **T4 GPU**. Training takes about 5 minutes.

## 1. Get the code and data

In [ ]:
REPO_URL = "https://github.com/zainabali18/Mizan.git"   # your repository

!git clone -q $REPO_URL mizan_repo
%cd mizan_repo
!pip install -q -r requirements.txt transformers accelerate
!python -m mizan.data > /dev/null && echo "processed data ready"

In [ ]:
import json, time
import numpy as np, pandas as pd, torch
from sklearn.metrics import f1_score
from mizan.data import clean_text

labels = json.load(open("data/processed/labels.json", encoding="utf-8"))
label2id = labels["label2id"]
id2label = {int(k): v for k, v in labels["id2label"].items()}

train = pd.read_csv("data/processed/train.csv")
val   = pd.read_csv("data/processed/validation.csv")
test  = pd.read_csv("data/processed/test.csv")

# PII is redacted; letters are left natural because AraBERT was pretrained on natural Arabic.
for df in (train, val, test):
    df["x"] = [clean_text(t, normalise=False) for t in df.text]
print(len(train), len(val), len(test), "rows |", len(label2id), "intents")

## 2. Model and data loaders

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
set_seed(42)

MODEL_ID = "aubmindlab/bert-base-arabertv02"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=len(label2id), id2label=id2label, label2id=label2id)

class IntentDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tok(list(df.x), truncation=True, max_length=64)
        self.y = list(df.label_id)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = int(self.y[i])
        return item

train_ds, val_ds = IntentDataset(train), IntentDataset(val)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": (preds == p.label_ids).mean(),
            "macro_f1": f1_score(p.label_ids, preds, average="macro", zero_division=0)}

## 3. Train

Small data (about 14 examples per intent), so we train for more epochs and keep the checkpoint with the best **validation** macro-F1. The test set is not touched until the end.

In [ ]:
EPOCHS = 20
BATCH = 16
steps_per_epoch = -(-len(train_ds) // BATCH)            # ceiling division
args = TrainingArguments(
    output_dir="models/checkpoints",
    num_train_epochs=EPOCHS,
    learning_rate=5e-5,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=64,
    warmup_steps=int(0.1 * EPOCHS * steps_per_epoch),   # 10% warm-up (works on transformers 4.x and 5.x)
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    report_to="none",
    seed=42,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
t0 = time.time()
trainer.train()
TRAIN_MIN = (time.time() - t0) / 60
print(f"trained in {TRAIN_MIN:.1f} min | best validation:", trainer.evaluate())

## 4. Score on the test set (once)

In [ ]:
model.eval()

def predict_proba(texts, device):
    enc = tok(list(texts), truncation=True, max_length=64, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        return torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()

device = model.device
test_probs = np.vstack([predict_proba(test.x[i:i+64], device) for i in range(0, len(test), 64)])
val_probs  = np.vstack([predict_proba(val.x[i:i+64], device)  for i in range(0, len(val), 64)])
test_pred = [id2label[i] for i in test_probs.argmax(1)]
print("test accuracy:", np.mean(np.array(test_pred) == test.label.values))

## 5. Single-query latency on CPU and GPU

Measured end to end on raw text (PII redaction, cleaning, tokenising, predicting), one customer message at a time. CPU is the realistic deployment target for a small on-premise service.

In [ ]:
def single_query_ms(device, n=50):
    model.to(device)
    for t in test.text[:3]:                       # warm-up
        predict_proba([clean_text(t, normalise=False)], device)
    times = []
    for t in test.text[:n]:
        s = time.perf_counter()
        predict_proba([clean_text(t, normalise=False)], device)
        if device == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - s) * 1000)
    return float(np.mean(times))

GPU_MS = single_query_ms("cuda") if torch.cuda.is_available() else float("nan")
CPU_MS = single_query_ms("cpu")
print(f"single query: CPU {CPU_MS:.0f} ms | GPU {GPU_MS:.0f} ms")

## 6. Save the model and record the result

In [ ]:
import os
SAVE_DIR = "models/mizan-arabert"
trainer.save_model(SAVE_DIR)
tok.save_pretrained(SAVE_DIR)
SIZE_MB = sum(os.path.getsize(os.path.join(SAVE_DIR, f)) for f in os.listdir(SAVE_DIR)) / 1e6

from mizan.evaluate import evaluate
result = evaluate(
    "Fine-tuned AraBERT",
    test.label.tolist(), test_pred, test.variety.tolist(),
    latency_ms=CPU_MS, size_mb=SIZE_MB,
    notes=(f"{MODEL_ID}, {EPOCHS} epochs on 1,078 examples ({TRAIN_MIN:.1f} min on T4), best epoch by "
           f"validation macro-F1; latency = single query on CPU (GPU: {GPU_MS:.0f} ms)"),
)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 7. Human handoff: when should the model ask for help?

A bank should not auto-answer when the model is unsure. On the **validation** set we pick the lowest confidence threshold at which auto-handled requests are at least 90% accurate, then check how that threshold behaves on the **test** set.

In [ ]:
TARGET = 0.90
val_pred_ids, val_conf = val_probs.argmax(1), val_probs.max(1)
val_true_ids = val.label_id.values

THRESHOLD = 0.95
for th in np.round(np.arange(0.30, 0.96, 0.05), 2):
    keep = val_conf >= th
    if keep.any() and (val_pred_ids[keep] == val_true_ids[keep]).mean() >= TARGET:
        THRESHOLD = float(th); break

test_conf = test_probs.max(1)
auto = test_conf >= THRESHOLD
auto_acc = float((np.array(test_pred)[auto] == test.label.values[auto]).mean()) if auto.any() else 0.0
handoff = {
    "confidence_threshold": THRESHOLD,
    "test_auto_handled_share": round(float(auto.mean()), 4),
    "test_auto_handled_accuracy": round(auto_acc, 4),
    "test_routed_to_human_share": round(float(1 - auto.mean()), 4),
}
print(json.dumps(handoff, indent=2))

In [ ]:
config = {"model_id": SAVE_DIR, "base_model": MODEL_ID, "max_length": 64, "normalise": False, **handoff}
json.dump(config, open("mizan_config.json", "w"), indent=2)
config

## 8. (Recommended) Publish the model to the Hugging Face Hub

This lets the API, Docker image and live demo download the model, which is too big for GitHub (about 540 MB).
1. Create a free account at huggingface.co.
2. Go to Settings → Access Tokens and create a token with **write** access.
3. Run the cell below and paste the token when asked.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HUB_ID = "zainabali18/mizan-arabert-arbanking77"   # <your-hf-username>/mizan-arabert-arbanking77
model.to("cpu")
model.push_to_hub(HUB_ID)
tok.push_to_hub(HUB_ID)
config["model_id"] = HUB_ID
json.dump(config, open("mizan_config.json", "w"), indent=2)
print("published:", f"https://huggingface.co/{HUB_ID}")

## 9. Download the results into your repository

In [ ]:
from google.colab import files
files.download("results/fine_tuned_arabert.json")   # -> results/
files.download("mizan_config.json")                 # -> repository root